# Análisis de viabilidad del TFM
En `df_fases_MES`tengo la **fase**, **máquina dominante**, y el **grupo/familia de esa máquina**. Con eso puedo montar un  cruce fase × máquina dentro de cada familia, ahi es donde vive la decisión de asignación.

Entonces, la elección real no es "esta fase fue a esta máquina" (eso siempre pasa). La elección existe cuando fases del mismo tipo pudieron ir a máquinas distintas de la misma familia.   
Por eso el análisis interesante es dentro de cada grupo: `¿las fases de una familia se reparten entre sus varias máquinas, o cada tipo de fase va siempre a la misma?`

In [23]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow as pa

In [24]:
df_fases_MES = pd.read_parquet("res//procesed/df_fases_MES.parquet")
display(df_fases_MES.head(1))

,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,n_imputaciones,CodigoGrupo,NombreGrupo
0,00013C31-3895-4370-BB3E-4255371E8D8F,TN09,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.0,2,TPU,Torns Puma (Taller COURES)


Cuántas máquinas y cuántas fases tiene cada familia:

In [25]:
resumen_familia = df_fases_MES.groupby("NombreGrupo").agg(
    n_fases=("SmFaseUid", "nunique"),
    n_maquinas=("MaquinaDominante", "nunique"),
).sort_values("n_fases", ascending=False)

print(resumen_familia)

                                          n_fases  n_maquinas
NombreGrupo                                                  
Torns Puma (Taller COURES)                   4384           3
Màquines de Taladrat Profund                 3996           2
Centres de Mecanitzat Vertical COURES        3269           4
Premses Conformats                           3122           4
Torns Control Numèric                        2301           4
Centres de Mecanitzat Horitzontal COURES     1644           3
Centre de Mecanitzat 5 Eixos petites         1406           4
Centres de Mecanitzat Vertical ALUMINIS       830           3
Màquines Secció Flexibles                     254           4


## Decisión de Viabilidad 
Tras todas las limpiezas que he realizado tengo ya un bloque sólido de famílias, tengo 8 familias con elección real y volumen.

Excepto por las familias: 
- Màquines Secció Flexibles: presenta 254 fases en 4 máquinas, siendo poco volumen. Repartido entre 4 máquinas y luego entre tipos de fase, puede quedarse corta para conclusiones robustas. La mantengo por ahora, pero con la etiqueta mental de "``frágil``".



Toca observar si las imputaciones se reparten entre las máquinas de la família.

In [28]:
familias_analizables = [
    "Torns Puma (Taller COURES)",
    "Màquines de Taladrat Profund",
    "Centres de Mecanitzat Vertical COURES",
    "Premses Conformats",
    "Torns Control Numèric",
    "Centres de Mecanitzat Horitzontal COURES",
    "Centre de Mecanitzat 5 Eixos petites",
    "Centres de Mecanitzat Vertical ALUMINIS",
    "Màquines Secció Flexibles",
]

for familia in familias_analizables:
    sub = df_fases_MES[df_fases_MES["NombreGrupo"] == familia]
    reparto = sub["MaquinaDominante"].value_counts(normalize=True)
    print(f"{familia[:45]:45s} | fases: {len(sub):5d} | máq. principal: {reparto.iloc[0]:.0%} | nº máq. usadas: {(reparto > 0.01).sum()}")

Torns Puma (Taller COURES)                    | fases:  4384 | máq. principal: 78% | nº máq. usadas: 2
Màquines de Taladrat Profund                  | fases:  3996 | máq. principal: 87% | nº máq. usadas: 2
Centres de Mecanitzat Vertical COURES         | fases:  3269 | máq. principal: 43% | nº máq. usadas: 4
Premses Conformats                            | fases:  3122 | máq. principal: 70% | nº máq. usadas: 4
Torns Control Numèric                         | fases:  2301 | máq. principal: 33% | nº máq. usadas: 4
Centres de Mecanitzat Horitzontal COURES      | fases:  1644 | máq. principal: 69% | nº máq. usadas: 3
Centre de Mecanitzat 5 Eixos petites          | fases:  1406 | máq. principal: 59% | nº máq. usadas: 4
Centres de Mecanitzat Vertical ALUMINIS       | fases:   830 | máq. principal: 63% | nº máq. usadas: 3
Màquines Secció Flexibles                     | fases:   254 | máq. principal: 93% | nº máq. usadas: 3


Este es el resultado que cierra la viabilidad y la conclusión es positiva.
- **Torns Control Numèric** — 33% principal, 4 máquinas. Excelente. El trabajo está muy repartido entre las 4 — justo lo que valida la fusión que hiciste, y el mejor ejemplo de elección real. 2.301 fases.
- **Centres Vertical COURES** — 43% principal, 4 máquinas. Reparto sano sobre 3.269 fases. Familia estrella.
- **Centre 5 Eixos petites** — 59% principal, 4 máquinas. Buen reparto, 1.406 fases.
- **Centres Vertical ALUMINIS** — 63% principal, 3 máquinas. Correcto, 830 fases.
- **Centres Horizontal COURES** — 69% principal, 3 máquinas. Aceptable, 1.644 fases.
- **Premses Conformats** — 70% principal, 4 máquinas. La principal pesa, pero las otras 3 tienen uso real. Válida, 3.122 fases.
> Estas 6 son un núcleo sólido, hay decisión de asignación efectiva y volumen de sobra. 

- **Torns Puma** — 78% principal, solo 2 de 3 máquinas usadas. Ojo: tiene 3 máquinas en el grupo pero una casi no se usa. Aun así, 4.384 fases y un 22% en la segunda máquina da margen. Utilizable, con matiz.
- **Màquines de Taladrat Profund** — 87% principal, solo 2 de 2. Una máquina hace casi todo. Elección limitada, pero con 3.996 fases el 13% restante son ~500 fases en la otra. 
> Estas 2 son frontera, en principio contaré con ellas, pero debo estar dispuesto a descartarlas según como evolucione.

- **Màquines Secció Flexibles** — 93% principal, y solo 254 fases. Aquí sí: una máquina acapara casi todo y el volumen es bajo. La elección es prácticamente ficticia. Yo la excluiría del análisis de idoneidad, o la mencionaría explícitamente como familia sin elección efectiva. No aporta.Màquines Secció Flexibles — 93% principal, y solo 254 fases. Aquí sí: una máquina acapara casi todo y el volumen es bajo. La elección es prácticamente ficticia. Yo la excluiría del análisis de idoneidad, o la mencionaría explícitamente como familia sin elección efectiva. No aporta.

## Conclusión
De las familias productivas con múltiples máquinas, la mayoría presentan un reparto efectivo del trabajo entre sus máquinas (con la principal concentrando entre el 33% y el 70% de las fases), lo que confirma que la asignación fase-máquina constituye una decisión real y no determinista en el taller. Esto valida empíricamente la premisa central del TFM: existe un problema de asignación con elección genuina que un modelo de idoneidad puede aprender a optimizar.

NOTA PERSONAL:
puede que haya criterios legítimos (tamaño de pieza, disponibilidad, precisión requerida) que hacen que cierta máquina sea la natural para la mayoría de fases, reservando las otras para casos concretos. Eso también es idoneidad, y de hecho es lo interesante que tu modelo podría capturar. Así que no descartes las de principal alta a la ligera; el matiz es "poca elección" en Secció Flexibles (por volumen + concentración), no en las demás.